### 1. Trap - Mutable default argument — the most common trap

In [38]:
def append_to(item, result=[]):     # create func obj & eval defaults(once) & store in func.__defaults__
    result.append(item)             # body is compiled to bytecode but not executed
    return result

**every call that omits `result` receives that same list object — not a fresh one**

In [4]:
append_to(1)

[1]

In [5]:
append_to(2)

[1, 2]

In [6]:
append_to(3)

[1, 2, 3]

The `def` statement does three things when Python reaches it:

1. Creates the function object
2. Evaluates all default value expressions — right then, once
3. Stores those evaluated values on the function object (`func.__defaults__`)

The body is compiled to bytecode but **not executed**. It only runs when the function is called.

So for the mutable default:

```python
def append_to(item, result=[]):   # ← [] is evaluated HERE, once
    result.append(item)           # ← these lines are NOT touched yet
    return result
```

When Python hits the `def` line, it creates one list object and stores it. You can see it sitting there before any call:

```python
append_to.__defaults__   # ([],)  ← the list already exists
```

Then every call that omits `result` receives that **same** list object — not a fresh one.

The distinction that matters for interviews:

```python
# default evaluated at def time (once)
def f(x=[]):    ...

# default evaluated at call time (each call)
def f(x=None):
    if x is None:
        x = []  # new list created on every call
```

The `None` sentinel pattern works precisely because the `[]` creation is moved **inside the body**, which runs on every call instead of once at definition.

In [37]:
def append_to(item, result: list | None = None) -> list:  # Fix
    if result is None:
        result = []
    result.append(item)
    return result

Example 2

In [50]:
def func(x=[]):   # 1) creates func obj; 2) evaluates default values (ONCE); 3) stores defaults into func.__defaluts__ = ([],)
    x.append(1)   # [1]
    print(x)

In [51]:
func()  # when default 'x' is omitted, it recieves the default obj from func.__defaults__ -> []

[1]


In [52]:
func([1, 2])  # uses new list obj that is being passed in, not default one

[1, 2, 1]


In [53]:
func()  # uses the default list again -> [1]

[1, 1]


In [46]:
func([2, 3])

[2, 3, 1]


### 2. Trap - Immutable Defaults

In [ ]:
def f(x=0):     # default is never touched
    x += 1      # creates a new int object and rebinds the local x
    return x    # same for str, float, bool, bytes

f()  # 1
f()  # 1  — always 1

#### Trick 1 — `+=` behaves differently on mutables vs immutables

In [ ]:
def f(x=[]):
    print("x before +=: ", id(x))
    x += [1]                       # calls x.__iadd__([1]) — mutates in place
    print("x after +=: ", id(x))
    return x

In [61]:
f()  # [1]

x before +=:  1680917734400
x after +=:  1680917734400


[1]

- `+=` on a list calls `__iadd__`, which **mutates in place**
- `+` creates a **new object**

In [69]:
def f(x=[]):
    print("x before +[1]: ", id(x))
    x = x + [1]                    # creates a NEW list, rebinds local x
    print("x after +[1]: ", id(x))
    return x

In [70]:
f()  # [1]

x before +[1]:  1680917759296
x after +[1]:  1680917901440


[1]

In [71]:
f()  # [1]  — safe, but for the wrong reason: the default is still mutable

x before +[1]:  1680917759296
x after +[1]:  1680917771904


[1]

For `int` and `str`, there is no `__iadd__` that mutates — the result is always a new object regardless of which form you use.

---

#### Trick 2 — tuple as default with mutable contents

A tuple default looks safe but can bite you if the tuple contains mutables:

In [72]:
def f(data=([],)):
    data[0].append(1)  # mutating the list inside the shared tuple
    return data

In [73]:
f()  # ([1],)

([1],)

In [74]:

f()  # ([1, 1],)  ← the list inside the default tuple is shared across calls

([1, 1],)

### 3. Trap - Late binding closures — the loop trap

In [ ]:
funcs = [lambda: i for i in range(3)]  # there are 2 scopes - lambda local, comprehension

#### Iteration 1 — `i = 0`

- Python creates `lambda: i` — a function object
- This lambda has **no parameters** and **no local variables**
- `i` it references **is not inside the `lambda`** — it's in the comprehension scope above it.
- The lambda stores a **reference** to that scope's `i` variable, not to its current **value** `0`.
```py
funcs = [<lambda_1>]       # lambda_1 closes over → i (the variable)
i = 0
```
At this exact moment, calling `lambda_1()` would return `0`. But nobody is calling it yet.

---

#### Iteration 2 — `i = 1`

Python reassigns `i`. The same memory slot now holds `1`. No new `i` is created — the existing one is updated.
```python
funcs = [<lambda_1>, <lambda_2>]
i = 1                        # same variable, new value
```
- Now `lambda_1` still closes over the same `i`. Calling it now would return `1`. It "forgot" about `0` — because it never remembered `0`. It only remembers where `i` lives, not what it was.
- `lambda_2` also closes over the exact same `i`.

---

#### Iteration 3 — `i = 2`
```python
funcs = [<lambda_1>, <lambda_2>, <lambda_3>]
i = 2                        # same variable, updated again
```
**All three lambdas point to the same `i`**. Calling any of them now returns `2`.

In [78]:
[f() for f in funcs]   # [2, 2, 2], not [0, 1, 2]

[2, 2, 2]

The lambda body isn't evaluated when the lambda is created — it's evaluated when called. By then, the loop has finished and `i` is `2`. All three lambdas close over the **same** `i` variable (not its value at creation time).

---

#### What the closure actually stores — you can inspect it
Python implements closed-over variables as **cell objects** — a small wrapper that holds a reference to the current value. All three lambdas share one cell. When `i` changes, the cell updates, and all three lambdas see the change immediately.

In [82]:
funcs = [lambda: i for i in range(3)]

In [83]:
# All three share the SAME cell object (the same i)
funcs[0].__closure__[0] is funcs[1].__closure__[0]  # True

True

In [84]:
funcs[0].__closure__[0] is funcs[2].__closure__[0]  # True

True

In [85]:
# The cell holds the final value
funcs[0].__closure__[0].cell_contents   # 2

2

In [86]:
lam1 = funcs[0]
lam2 = funcs[1]
lam3 = funcs[2]

In [90]:
lam1.__closure__[0] == lam2.__closure__[0] == lam3.__closure__[0]

True

In [91]:
lam1.__closure__[0].cell_contents

2

### 4. Trap — def inside a for loop (not just lambda)

Late binding is not a lambda-specific or comprehension-specific trap — it applies to any function created inside a loop that closes over the loop variable.

In [8]:
# Not just lambda — def inside a loop has the SAME late binding trap
funcs = []
for i in range(3):
    def f():
        return i        # closes over the same 'i' in the enclosing scope
    funcs.append(f)

In [9]:
print([f() for f in funcs])   # [2, 2, 2]  ← same result as the lambda version

[2, 2, 2]


In [3]:
# ✅ Same fixes apply
funcs = []
for i in range(3):
    def f(i=i):         # default arg captures the current value at def time
        return i
    funcs.append(f)

In [4]:
print([f() for f in funcs])   # [0, 1, 2]  ✓

[0, 1, 2]


**`def` in a `for` loop** — the exact same late binding problem as the lambda/comprehension version, just with named functions. The root cause is identical: `f` closes over `i` in the enclosing scope.

---

### 5. Trap — for loop variable leaks, comprehension scope doesn't

In Python 3, list/dict/set comprehensions have their **own scope** — the loop variable stays inside.
A `for` loop has **no scope** — the loop variable lives in the enclosing scope and remains accessible after the loop ends.

In [13]:
# List comprehension — 'j' is scoped INSIDE the comprehension (Python 3)
funcs = [lambda: j for j in range(3)]
print(j)  # NameError — 'j' does not leak out of the comprehension

NameError: name 'j' is not defined

In [14]:
# for loop — 'i' lives in the ENCLOSING scope, alive after the loop ends
funcs2 = []
for i in range(3):
    funcs2.append(lambda: i)

In [15]:
print(i)                        # 2  ← 'i' leaked out, still accessible
print([f() for f in funcs2])    # [2, 2, 2]  ← same late binding trap as comprehension

2
[2, 2, 2]


In [17]:
# Practical consequence: accidental reuse of loop variable
total = 0
for i in range(5):
    total += i

print(f"total={total}, i={i}")  # total=10, i=4  ← 'i' is still 4

total=10, i=4


**scope difference between `for` loop and comprehension** — `for` loops have no scope, so `i` leaks into the enclosing namespace and stays alive after the loop. List/dict/set comprehensions have their own scope in Python 3 — `i` disappears when the comprehension finishes. Practical trap: accidentally reusing a loop variable after the loop.

---

### 6. Trap — dict of functions from a loop

The late binding trap in its most practical form: building a mapping of callbacks, event handlers, or routes in a loop.
Every value in the dict ends up calling the same final loop variable.

In [20]:
# ❌ all lambdas close over the same 'action' variable
handlers = {}
for action in ["save", "load", "quit"]:
    handlers[action] = lambda: action

print(handlers["save"]())   # "quit"  ← wrong
print(handlers["load"]())   # "quit"  ← wrong

quit
quit


In [21]:
# ✅ Fix 1 — default argument captures value at creation
handlers = {action: (lambda a=action: a) for action in ["save", "load", "quit"]}
print(handlers["save"]())   # "save"  ✓

save


In [23]:
# ✅ Fix 2 — factory function: each call creates a NEW scope with its own 'action'
def make_handler(action: str):
    def handler():
        return action   # closes over the LOCAL 'action' parameter, not a shared variable
    return handler

handlers = {action: make_handler(action) for action in ["save", "load", "quit"]}
print(handlers["save"]())   # "save"  ✓
print(handlers["load"]())   # "load"  ✓

save
load


**dict of functions from a loop** — the late binding trap in its most realistic form. Building handlers, callbacks, or routes in a loop where each value should capture a different key. Two clean fixes shown: default-arg capture and the factory function pattern (each factory call creates an independent scope with its own local `action`).

---

### 7. Trap — `functools.partial` captures value, lambda does not

`partial(func, arg)` freezes `arg` **at creation time** — it becomes a local inside `partial`'s internal closure.
A `lambda` that references an outer variable just closes over the variable, not the value.
Use `partial` when you want to pre-fill an argument from a loop variable.

In [ ]:
from functools import partial

def log(level: str, msg: str) -> None:
    print(f"[{level}] {msg}")

# ❌ lambda — closes over 'lvl' variable, not its current value
loggers_bad = {lvl: lambda msg: log(lvl, msg) for lvl in ["INFO", "WARN", "ERROR"]}
loggers_bad["INFO"]("connected")    # [ERROR] connected  ← wrong
loggers_bad["WARN"]("connected")    # [ERROR] connected  ← wrong

# ✅ partial — binds log + the current value of lvl at creation time
loggers_good = {lvl: partial(log, lvl) for lvl in ["INFO", "WARN", "ERROR"]}
loggers_good["INFO"]("connected")   # [INFO] connected   ✓
loggers_good["WARN"]("low memory")  # [WARN] low memory  ✓

**`functools.partial` vs lambda** — `partial(log, lvl)` binds `lvl` immediately as an argument, so it's not a closure variable at all. A lambda that references `lvl` closes over the variable. The practical rule: when you want to pre-fill an argument from a loop variable, `partial` is safer than lambda.